# 03. 앙상블 학습 — 분류 실습 (RandomForest · XGBoost · Stacking)
> 교재 92~113p · SK네트웍스 Family AI 캠프 37기 · 2026-10-14

**앙상블(Ensemble)**: 여러 개의 모델을 연결해 더 강력한 모델을 만드는 기법

| 방식 | 핵심 | 대표 모델 |
|---|---|---|
| **Bagging** | 복원 추출한 서브 데이터로 모델 여러 개를 **병렬** 학습 → 투표/평균 | RandomForest |
| **Boosting** | 이전 모델의 오류(잔차)를 다음 모델이 보완하는 **순차** 학습 | XGBoost |
| **Stacking** | 개별 모델의 예측값을 새 특성으로 만들어 **메타 모델**이 최종 예측 | StackingClassifier |

**분류 문제의 투표**
- Hard Voting: 각 모델이 예측한 클래스의 **다수결**
- Soft Voting: 클래스별 예측 **확률의 평균** → 가장 높은 클래스 (보통 더 우수)

In [ ]:
# 필요한 라이브러리 임포트
import pandas as pd

# 경고 메시지 숨기기
import warnings
warnings.filterwarnings('ignore')

## 1. 데이터 불러오기 — 당뇨병 건강 지표 (BRFSS 2015, 50:50 split)
- 정답(target): `Diabetes_binary` (0: 정상, 1: 당뇨/전당뇨)
- 특성: 고혈압, 고지혈증, BMI, 흡연, 나이 등 21개 / 70,692행

In [ ]:
# 노트북과 같은 폴더에 CSV 파일을 두고 실행 (다른 위치라면 경로만 수정)
file_path = './diabetes_binary_5050split_health_indicators_BRFSS2015.csv'

df = pd.read_csv(file_path)
print(f'데이터 크기: {df.shape}')   # (70692, 22)
display(df.head())

## 2. 학습용 / 평가용 데이터 생성 (80 : 20)

In [ ]:
from sklearn.model_selection import train_test_split

X_data = df.drop(columns=['Diabetes_binary'])
y_data = df['Diabetes_binary']

# stratify=y_data : 학습용/평가용의 정답 비율(0:1)을 원본과 동일하게 유지
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data,
    test_size=0.2,
    random_state=0,
    stratify=y_data
)

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

In [ ]:
# stratify 효과 확인: 원본 / 학습용 / 평가용의 정답 비율이 모두 같아야 함
ratio = pd.DataFrame({
    '원본': y_data.value_counts(normalize=True),
    '학습용': y_train.value_counts(normalize=True),
    '평가용': y_test.value_counts(normalize=True),
})
display(ratio.round(3))

## 3. RandomForest 분류 모델 (Bagging)
- 결정 트리를 기본 모델로 사용하는 대표적 배깅 앙상블
- 샘플(행)은 복원 추출, 노드를 나눌 때는 특성(열)도 무작위로 일부(√n_features)만 검토 → **랜덤 패치**
- `n_estimators` 기본값 = 100 (트리 100개)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 모델 생성 (클래스 이름은 대문자로 시작)
rf = RandomForestClassifier(random_state=0)

# 모델 학습
rf.fit(X_train, y_train)

# 평가용 데이터를 이용한 예측 (정답 없이 X_test만 넣는다)
pred_test_rf = rf.predict(X_test)

# 모델 평가: accuracy_score(정답, 예측값)
acc_rf = accuracy_score(y_test, pred_test_rf)
print(f'RandomForest 정확도: {acc_rf:.4f}')

## 4. XGBoost 분류 모델 (Boosting)
- '조각상 다듬기': 큰 덩어리(평균) → 큰 오차 깎기 → 미세한 오차 깎기 … 를 수백 번 반복
- 각 단계의 트리는 **이전까지의 잔차(오차)** 를 학습하고, `학습률 × 예측`만큼 반영
- 설치: `pip install xgboost`
- ⚠️ 교재의 `XGBClassifer`는 오타 → 올바른 이름은 **`XGBClassifier`**

In [ ]:
from xgboost import XGBClassifier

# 모델 생성 · 학습
xgb = XGBClassifier(random_state=0)
xgb.fit(X_train, y_train)

# 예측 · 평가
pred_test_xgb = xgb.predict(X_test)
acc_xgb = accuracy_score(y_test, pred_test_xgb)
print(f'XGBoost 정확도: {acc_xgb:.4f}')

## 5. Stacking 분류 모델
**학습 2단계**: [기본 모델들 train] → [출력값 + 정답 결합] → [메타 모델 train]
**예측 2단계**: [기본 모델들 predict] → [출력값 결합] → [메타 모델 predict]

> `StackingClassifier`는 내부적으로 교차 검증(cv=5)으로 기본 모델의 예측값을 만들어
> 메타 모델이 '이미 본 데이터의 예측'으로 학습하는 과대적합을 막는다.

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

# 1단계: 개별 기본 모델(Base Estimators) — (이름, 모델) 튜플의 리스트
base_estimators = [
    ('rf', RandomForestClassifier(random_state=0)),
    ('xgb', XGBClassifier(random_state=0)),
    ('dt', DecisionTreeClassifier(random_state=0)),
]

# 2단계: 최종 메타 모델(Meta Learner / Blender)
final_meta_learner = LogisticRegression(max_iter=1000, random_state=0)

# Stacking 모델 생성
stack = StackingClassifier(
    estimators=base_estimators,
    final_estimator=final_meta_learner,
    n_jobs=-1          # CPU 코어를 모두 사용해 학습 시간 단축
)

In [ ]:
# 모델 학습 (기본 모델 3개 × 교차검증 5회 -> 시간이 조금 걸림)
stack.fit(X_train, y_train)

# 예측 · 평가
pred_test_stack = stack.predict(X_test)
acc_stack = accuracy_score(y_test, pred_test_stack)
print(f'Stacking 정확도: {acc_stack:.4f}')

## 6. 결과 비교

In [ ]:
result = pd.DataFrame({
    'model': ['RandomForest', 'XGBoost', 'Stacking'],
    'accuracy': [acc_rf, acc_xgb, acc_stack],
}).set_index('model').sort_values('accuracy', ascending=False)

display(result.round(4))

## ✅ 정리
- 이번 실습 기준 정확도: RandomForest ≈ 0.740 · XGBoost ≈ 0.750 · Stacking ≈ 0.748
- Boosting(XGBoost)이 Bagging(RandomForest)보다 약간 높았고,
  Stacking이 항상 최고는 아니다 → **기본 모델들이 서로 다른 실수를 할 때** 효과가 크다
- 분류 평가는 `accuracy_score`, 정답 비율 유지는 `stratify`